# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR^2 dataset package using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, referencing all record sets, fields, and columns by their `@id` fields as per the Croissant specification.

### Dataset Source
(Croissant schema URL): [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)  
This is a tabular dataset of 77 cancer survivors with second primary colorectal cancer, including clinical and pathological variables such as demographics, comorbidities, primary/secondary cancer types, treatment history, diagnosis intervals, anatomical CRC location, histopathology, presence of metastasis, and microsatellite instability status (MSI/MMR).

In [ ]:
# Ensure mlcroissant is installed
!pip install --quiet mlcroissant

## 1. Data Loading
Load dataset metadata and available records from the Croissant schema using `mlcroissant`. The dataset schema URL uniquely identifies the resource.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Title:", metadata.name)
print("Description:", metadata.description)
print("Published:", metadata.datePublished)
print("Version:", metadata.version)

## 2. Data Overview

Review the available **record sets** in the dataset, as well as the **fields** and **columns** associated with them. All entities are identified by their Croissant `@id` fields.

Below, we list the available record sets, then for each record set, its fields and their `@id`s. Finally, we'll preview several records from each set using `mlcroissant`.

In [ ]:
# List all available RecordSets in the dataset by @id
record_sets = list(dataset.record_sets)

print(f"Total record sets: {len(record_sets)}\n")
all_fields = {}
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    if hasattr(rs, 'name'):
        print(f"  name: {rs.name}")
    if hasattr(rs, 'description'):
        print(f"  description: {rs.description}")
    # Print associated fields with their @id
    if hasattr(rs, 'fields'):
        print("  Fields:")
        fields = rs.fields
        all_fields[rs.id] = fields
        for f in fields:
            print(f"    - {f.id}")
    print()
if not record_sets:
    print('No record sets found in this dataset. (This may be a metadata-only schema or the recordSets are named differently.)')

# For demonstration, preview the first few records of each RecordSet, if any exist.
for rs in record_sets:
    rs_id = rs.id
    print(f'\nPreview records for RecordSet @id: {rs_id}')
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        print(df.head(3))
    except Exception as e:
        print(f"  Could not load records for this RecordSet: {e}")

## 3. Data Extraction

Load one or more record sets into Pandas DataFrames for further analysis by **referencing their `@id` fields**.
***
**Note:** For this dataset, the main tabular data is often contained in a single record set – identify its `@id` from the overview above. Replace the placeholder if needed.
***

In [ ]:
# Example: Load all records from the primary tabular RecordSet
# Use the correct @id string of the main tabular record set.

# List all record set @ids
record_set_ids = [rs.id for rs in record_sets]
print('RecordSet IDs:', record_set_ids)

# For demonstration, try the first record set; update index as needed based on the previous code block's output
primary_record_set_id = record_set_ids[0] if record_set_ids else None

dataframes = {}
if primary_record_set_id:
    records = list(dataset.records(record_set=primary_record_set_id))
    df = pd.DataFrame(records)
    dataframes[primary_record_set_id] = df
    print(f"\n{primary_record_set_id} DataFrame columns:", df.columns.tolist())
    display(df.head())
else:
    print('No RecordSets available for extraction.')

## 4. Exploratory Data Analysis (EDA)

Let's examine the main DataFrame, select relevant columns **by their field `@id`**, and perform basic processing: filtering a numeric field (such as age), normalization, and grouping.

***
- **Set the appropriate `@id` values for numeric and grouping fields from the DataFrame columns.**
***

In [ ]:
import numpy as np

# Identify a numeric field by @id, such as age (replace below if column name differs)
if primary_record_set_id in dataframes:
    df = dataframes[primary_record_set_id]
    numeric_candidates = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric fields: {numeric_candidates}")

    # Choose a numeric field @id, e.g., '@id': 'age' (replace as needed)
    # If the numeric field has a different @id, select accordingly.
    numeric_field_id = numeric_candidates[0] if numeric_candidates else None
    group_field_id = None
    # Try to select a grouping field with string values, e.g., sex, anatomical_site, etc.
    for col in df.columns:
        if col.lower() in ['sex', 'gender', 'anatomical_location', 'msi_status']:
            group_field_id = col
            break
    if not group_field_id and df.columns.tolist():
        # Fallback to the first string column
        for col in df.columns:
            if df[col].dtype == object:
                group_field_id = col
                break
    if numeric_field_id:
        # Filtering records where value > threshold (use median if unsure)
        threshold = df[numeric_field_id].median() if not np.isnan(df[numeric_field_id]).all() else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold} (N={len(filtered_df)}):")
        display(filtered_df.head())

        # Normalize the numeric field (z-score)
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std(ddof=0)
        print(f"\nNormalized {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, norm_col]].head())

        # Group by the group field if available
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
    else:
        print('No numeric fields found for EDA.')
else:
    print('Primary DataFrame not available for EDA.')

## 5. Visualization

Let's plot a histogram of the selected numeric field (e.g., age or relevant measurement by its `@id`) and, if applicable, compare distributions across groups.

_All axes and legends will use the Croissant field `@id` values to comply with identifier referencing._

In [ ]:
import matplotlib.pyplot as plt

if primary_record_set_id in dataframes and numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    df = dataframes[primary_record_set_id]
    plt.hist(df[numeric_field_id].dropna(), bins=15, alpha=0.7)
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.title(f'Distribution of {numeric_field_id}')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8, 4))
        for g in df[group_field_id].dropna().unique():
            plt.hist(df[df[group_field_id]==g][numeric_field_id].dropna(), bins=8, alpha=0.6, label=str(g))
        plt.xlabel(numeric_field_id)
        plt.ylabel('Count')
        plt.title(f'{numeric_field_id} Distribution by {group_field_id}')
        plt.legend(title=group_field_id)
        plt.show()
else:
    print('Not enough data available for visualization.')

## 6. Conclusion

- In this notebook, we've demonstrated end-to-end loading, schema exploration, and analysis of the FAIR^2 colorectal cancer survivors dataset via its Croissant schema and the `mlcroissant` library.
- All data access and visualizations referenced fields and record sets by their Croissant `@id` values for reproducibility and schema compliance.
- You can extend this workflow to investigate relationships between molecular markers (e.g., MSI status), anatomical site, and clinical outcomes using the clean structured data.

> _For further analysis, consult the Croissant schema for more detailed field descriptions, and refer to the [mlcroissant documentation](https://pypi.org/project/mlcroissant/) for advanced API usage._